In [ ]:
"""
A2A Protocol (Agent-to-Agent) Implementation
================================================

Complete implementation of the Agent-to-Agent protocol for decentralized 
AI experimentation with depth limiting and discovery logging.

SCENARIO: Funder Intelligence System
====================================

International Development Organization using AI to optimize funding strategy
through three autonomous agents that collaborate via A2A protocol:

1. Fundraising Agent
   - Intelligence on individual high-net-worth funders
   - Portfolio data, investment interests, capacity, commitment levels
   - Geographic focus and thematic priorities
   - Decision timelines and past investment patterns

2. Business Development Agent
   - RFP (Request for Proposal) tracking and analysis
   - Competitive landscape and bidding information
   - Funder priorities and evaluation criteria
   - Win rates and budget availability by sector/region

3. Field Operations Agent
   - Local market intelligence and demand assessment
   - Current project performance metrics and team capacity
   - Local partnerships and government relationships
   - Stakeholder priorities and partner ecosystem

During the 3-month discovery phase, agents call each other autonomously
to gather funding intelligence, with depth limiting preventing cascading
failure while revealing actual workflow patterns.

COMPONENTS:
===========

1. A2A Protocol Message Classes
   - MessageType, ResponseStatus enums
   - A2ARequest, A2AResponse, A2AMetadata, A2APayload
   - Standardized message format for agent communication

2. Agent Wrapper Library
   - DiscoveryBackend interface (mock implementation provided)
   - A2ACallLog for centralized logging and discovery analysis
   - A2AAgent main wrapper with call_agent method

3. Service Factory
   - create_agent_app() factory for spinning up FastAPI services
   - Automatic agent registration, health checks, statistics

4. Three Concrete Agent Examples
   - Fundraising Intelligence: investor profiles, capacity, interests
   - Business Development Intelligence: RFP data, competitive landscape
   - Field Operations Intelligence: local capacity, project performance, demand

5. Funding Strategy Logic
   - Demonstrates cascade calling across all three agents
   - Shows depth limiting in action
   - Illustrates workflows that hit constraints

USAGE:
======

Each agent team implements their own A2A service using the factory:

    app, agent = create_agent_app(
        agent_id="my-agent",
        description="What this agent does",
        discovery_backend=discovery,
        agent_logic_func=my_agent_logic,
        max_depth=2
    )

    # Run with: uvicorn app --port 8001

Agents call each other via the A2A protocol:

    response = await agent.call_agent(
        goal="what you want from another agent",
        parameters={"key": "value"},
        current_depth=0
    )

All calls are logged for 3-month discovery analysis:
- Call graph (who calls whom)
- Thwarted cascades (what hits depth limit)
- Performance metrics (latency, success rate)
- Workflow patterns (intended calling sequences)

After 3 months, orchestrator design is informed by actual patterns.
"""

import json
import uuid
import httpx
import asyncio
from datetime import datetime
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Any
from enum import Enum
import logging
from abc import ABC, abstractmethod

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ============================================================================
# PART 1: A2A PROTOCOL MESSAGE CLASSES
# ============================================================================

class MessageType(str, Enum):
    """A2A message types"""
    AGENT_CALL = "agent_call"
    AGENT_RESPONSE = "agent_response"
    AGENT_ERROR = "agent_error"


class ResponseStatus(str, Enum):
    """Response status codes"""
    SUCCESS = "success"
    ERROR = "error"
    DEPTH_LIMIT_EXCEEDED = "depth_limit_exceeded"
    NOT_FOUND = "not_found"
    TIMEOUT = "timeout"
    INVALID_REQUEST = "invalid_request"


@dataclass
class A2AMetadata:
    """Metadata for all A2A messages"""
    caller_id: str
    call_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    timestamp: str = field(default_factory=lambda: datetime.utcnow().isoformat() + "Z")
    call_depth: int = 0
    max_depth: int = 2
    trace_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class A2APayload:
    """Payload for agent calls"""
    goal: str
    parameters: Dict[str, Any] = field(default_factory=dict)
    context: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class A2ADiscovery:
    """Discovery metadata for how target agent was found"""
    method: str  # "vector_db_query", "registry_lookup", etc.
    query_embedding: Optional[List[float]] = None
    similarity_score: Optional[float] = None
    target_agent: Optional[str] = None
    alternatives: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class A2AResponseContract:
    """Expected response contract"""
    expected_schema: str = "generic"
    timeout_ms: int = 5000
    required_fields: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class A2ARequest:
    """Complete A2A request message"""
    version: str = "a2a-v1"
    message_type: MessageType = MessageType.AGENT_CALL
    metadata: Optional[A2AMetadata] = None
    payload: Optional[A2APayload] = None
    discovery: Optional[A2ADiscovery] = None
    response_contract: Optional[A2AResponseContract] = None

    def __post_init__(self):
        if self.metadata is None:
            self.metadata = A2AMetadata(caller_id="unknown")
        if self.payload is None:
            self.payload = A2APayload(goal="")
        if self.discovery is None:
            self.discovery = A2ADiscovery(method="manual")
        if self.response_contract is None:
            self.response_contract = A2AResponseContract()

    def to_dict(self) -> Dict:
        return {
            "version": self.version,
            "message_type": self.message_type.value,
            "metadata": self.metadata.to_dict(),
            "payload": self.payload.to_dict(),
            "discovery": self.discovery.to_dict(),
            "response_contract": self.response_contract.to_dict()
        }

    @classmethod
    def from_dict(cls, data: Dict) -> "A2ARequest":
        """Reconstruct A2ARequest from dict"""
        return cls(
            version=data.get("version", "a2a-v1"),
            message_type=MessageType(data.get("message_type", "agent_call")),
            metadata=A2AMetadata(**data.get("metadata", {})),
            payload=A2APayload(**data.get("payload", {})),
            discovery=A2ADiscovery(**data.get("discovery", {})),
            response_contract=A2AResponseContract(**data.get("response_contract", {}))
        )


@dataclass
class A2AResponse:
    """Complete A2A response message"""
    version: str = "a2a-v1"
    message_type: MessageType = MessageType.AGENT_RESPONSE
    metadata: Optional[Dict] = None
    status: ResponseStatus = ResponseStatus.SUCCESS
    result: Dict = field(default_factory=dict)
    errors: List[Dict] = field(default_factory=list)
    latency_ms: Optional[float] = None

    def to_dict(self) -> Dict:
        return {
            "version": self.version,
            "message_type": self.message_type.value,
            "metadata": self.metadata or {},
            "status": self.status.value,
            "result": self.result,
            "errors": self.errors,
            "latency_ms": self.latency_ms
        }

    @classmethod
    def from_dict(cls, data: Dict) -> "A2AResponse":
        """Reconstruct A2AResponse from dict"""
        return cls(
            version=data.get("version", "a2a-v1"),
            message_type=MessageType(data.get("message_type", "agent_response")),
            metadata=data.get("metadata", {}),
            status=ResponseStatus(data.get("status", "error")),
            result=data.get("result", {}),
            errors=data.get("errors", []),
            latency_ms=data.get("latency_ms")
        )


# ============================================================================
# PART 2: AGENT WRAPPER LIBRARY
# ============================================================================

class DiscoveryBackend(ABC):
    """Abstract interface for agent discovery backends"""
    
    @abstractmethod
    async def query(self, embedding: List[float], top_k: int = 3) -> List[Dict]:
        """Query for agents by embedding"""
        pass
    
    @abstractmethod
    async def register_agent(self, agent_metadata: Dict) -> None:
        """Register an agent"""
        pass
    
    @abstractmethod
    async def embed(self, text: str) -> List[float]:
        """Embed text (mock implementation for now)"""
        pass


class MockDiscoveryBackend(DiscoveryBackend):
    """Mock discovery backend for testing/demo"""
    
    def __init__(self):
        self.registry: Dict[str, Dict] = {}
    
    async def embed(self, text: str) -> List[float]:
        """Simple mock embedding (hash-based)"""
        # In production, use real embedding model
        hash_val = hash(text) % 1000
        return [float(hash_val % 10) / 10 for _ in range(384)]
    
    async def register_agent(self, agent_metadata: Dict) -> None:
        """Register agent in mock registry"""
        agent_id = agent_metadata.get("agent_id")
        self.registry[agent_id] = agent_metadata
        logger.info(f"Registered agent: {agent_id}")
    
    async def query(self, embedding: List[float], top_k: int = 3) -> List[Dict]:
        """Return top matching agents (mock: just return all)"""
        agents = list(self.registry.values())
        return agents[:top_k] if agents else []


class A2ACallLog:
    """Centralized call logging for discovery analysis"""
    
    def __init__(self):
        self.calls: List[Dict] = []
    
    def log_call(self, log_entry: Dict) -> None:
        """Log a single A2A call"""
        self.calls.append(log_entry)
        logger.info(f"A2A Call Logged: {log_entry['caller']} -> {log_entry['target']} "
                   f"[{log_entry['status']}] depth={log_entry['depth']}")
    
    def get_thwarted_calls(self) -> List[Dict]:
        """Get all calls blocked by depth limit"""
        return [c for c in self.calls if c['status'] == 'depth_limit_exceeded']
    
    def get_call_graph(self) -> Dict[str, List[str]]:
        """Extract calling patterns: who calls whom"""
        graph = {}
        for call in self.calls:
            if call['status'] == 'success':
                caller = call['caller']
                target = call['target']
                if caller not in graph:
                    graph[caller] = []
                if target not in graph[caller]:
                    graph[caller].append(target)
        return graph
    
    def get_stats(self) -> Dict:
        """Get summary statistics"""
        total_calls = len(self.calls)
        successful = len([c for c in self.calls if c['status'] == 'success'])
        blocked = len(self.get_thwarted_calls())
        errors = total_calls - successful - blocked
        
        avg_latency = 0
        latencies = [c['latency_ms'] for c in self.calls if c.get('latency_ms')]
        if latencies:
            avg_latency = sum(latencies) / len(latencies)
        
        return {
            "total_calls": total_calls,
            "successful": successful,
            "blocked_by_depth_limit": blocked,
            "errors": errors,
            "avg_latency_ms": round(avg_latency, 2),
            "thwarted_workflow_count": blocked
        }
    
    def export_for_analysis(self) -> Dict:
        """Export all data for analysis"""
        return {
            "calls": self.calls,
            "call_graph": self.get_call_graph(),
            "stats": self.get_stats(),
            "timestamp": datetime.utcnow().isoformat()
        }


class A2AAgent:
    """
    A2A Agent wrapper - handles discovery and calling other agents
    """
    
    def __init__(
        self,
        agent_id: str,
        description: str,
        discovery_backend: DiscoveryBackend,
        max_depth: int = 2,
        call_log: Optional[A2ACallLog] = None,
        endpoint: str = "http://localhost:8000"
    ):
        self.agent_id = agent_id
        self.description = description
        self.discovery_backend = discovery_backend
        self.max_depth = max_depth
        self.call_log = call_log or A2ACallLog()
        self.endpoint = endpoint
        self.http_client = httpx.AsyncClient(timeout=10.0)
    
    async def register(self) -> None:
        """Register this agent in discovery backend"""
        metadata = {
            "agent_id": self.agent_id,
            "description": self.description,
            "endpoint": self.endpoint,
            "max_depth": self.max_depth,
            "registered_at": datetime.utcnow().isoformat()
        }
        await self.discovery_backend.register_agent(metadata)
    
    async def call_agent(
        self,
        goal: str,
        parameters: Dict[str, Any] = None,
        current_depth: int = 0,
        trace_id: Optional[str] = None,
        context: Dict[str, Any] = None
    ) -> A2AResponse:
        """
        Call another agent discovered via the discovery backend
        
        Args:
            goal: What you want the other agent to do
            parameters: Parameters for the goal
            current_depth: Current call depth in the chain
            trace_id: Trace ID for distributed tracing
            context: Additional context
        
        Returns:
            A2AResponse with result or error
        """
        
        start_time = datetime.utcnow()
        trace_id = trace_id or str(uuid.uuid4())
        parameters = parameters or {}
        context = context or {}
        
        # Check depth limit BEFORE calling
        if current_depth >= self.max_depth:
            error_response = A2AResponse(
                metadata={
                    "caller_id": self.agent_id,
                    "trace_id": trace_id,
                    "timestamp": datetime.utcnow().isoformat()
                },
                status=ResponseStatus.DEPTH_LIMIT_EXCEEDED,
                errors=[{
                    "code": "depth_limit_exceeded",
                    "message": f"Cannot call agent for '{goal}' - depth limit {self.max_depth} reached",
                    "current_depth": current_depth,
                    "max_depth": self.max_depth
                }]
            )
            
            self.log_call(
                goal=goal,
                target=None,
                depth=current_depth,
                status=ResponseStatus.DEPTH_LIMIT_EXCEEDED.value,
                error=error_response.errors[0],
                latency_ms=None
            )
            return error_response
        
        # Discover target agent via embedding
        try:
            goal_embedding = await self.discovery_backend.embed(goal)
            candidates = await self.discovery_backend.query(goal_embedding, top_k=3)
        except Exception as e:
            logger.error(f"Discovery failed: {e}")
            return A2AResponse(
                status=ResponseStatus.ERROR,
                errors=[{
                    "code": "discovery_error",
                    "message": f"Failed to discover agents: {str(e)}"
                }]
            )
        
        if not candidates:
            error_response = A2AResponse(
                status=ResponseStatus.NOT_FOUND,
                errors=[{
                    "code": "no_agents_found",
                    "message": f"No agents found for goal: {goal}"
                }]
            )
            self.log_call(
                goal=goal,
                target=None,
                depth=current_depth,
                status=ResponseStatus.NOT_FOUND.value,
                error=error_response.errors[0]
            )
            return error_response
        
        # Best matching agent
        target_agent = candidates[0]
        target_agent_id = target_agent.get("agent_id", "unknown")
        target_endpoint = target_agent.get("endpoint", "")
        
        # Build A2A request
        request = A2ARequest(
            metadata=A2AMetadata(
                caller_id=self.agent_id,
                call_depth=current_depth,
                max_depth=self.max_depth,
                trace_id=trace_id
            ),
            payload=A2APayload(
                goal=goal,
                parameters=parameters,
                context=context
            ),
            discovery=A2ADiscovery(
                method="vector_db_query",
                target_agent=target_agent_id,
                alternatives=[c.get("agent_id") for c in candidates[1:]]
            )
        )
        
        # Call target agent
        try:
            response = await self.http_client.post(
                f"{target_endpoint}/a2a",
                json=request.to_dict(),
                timeout=5.0
            )
            
            latency_ms = (datetime.utcnow() - start_time).total_seconds() * 1000
            
            result_data = response.json()
            result = A2AResponse.from_dict(result_data)
            result.latency_ms = latency_ms
            
            self.log_call(
                goal=goal,
                target=target_agent_id,
                depth=current_depth,
                status=result.status.value,
                latency_ms=latency_ms
            )
            
            return result
        
        except httpx.TimeoutException:
            latency_ms = (datetime.utcnow() - start_time).total_seconds() * 1000
            error_response = A2AResponse(
                status=ResponseStatus.TIMEOUT,
                errors=[{
                    "code": "timeout",
                    "message": f"Agent {target_agent_id} did not respond within timeout",
                    "target": target_agent_id
                }],
                latency_ms=latency_ms
            )
            self.log_call(
                goal=goal,
                target=target_agent_id,
                depth=current_depth,
                status=ResponseStatus.TIMEOUT.value,
                latency_ms=latency_ms,
                error=error_response.errors[0]
            )
            return error_response
        
        except Exception as e:
            latency_ms = (datetime.utcnow() - start_time).total_seconds() * 1000
            error_response = A2AResponse(
                status=ResponseStatus.ERROR,
                errors=[{
                    "code": "call_error",
                    "message": f"Failed to call agent: {str(e)}",
                    "target": target_agent_id
                }],
                latency_ms=latency_ms
            )
            self.log_call(
                goal=goal,
                target=target_agent_id,
                depth=current_depth,
                status=ResponseStatus.ERROR.value,
                latency_ms=latency_ms,
                error=error_response.errors[0]
            )
            return error_response
    
    def log_call(
        self,
        goal: str,
        target: Optional[str],
        depth: int,
        status: str,
        error: Optional[Dict] = None,
        latency_ms: Optional[float] = None
    ) -> None:
        """Log an A2A call"""
        log_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "caller": self.agent_id,
            "goal": goal,
            "target": target,
            "depth": depth,
            "status": status,
            "latency_ms": latency_ms,
            "error": error
        }
        self.call_log.log_call(log_entry)
    
    def get_call_stats(self) -> Dict:
        """Get stats for this agent"""
        return self.call_log.get_stats()


# ============================================================================
# PART 3: SIMPLE AGENT SERVICE EXAMPLES
# ============================================================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


class A2ARequestBody(BaseModel):
    """Pydantic model for A2A requests"""
    version: str
    message_type: str
    metadata: Dict
    payload: Dict
    discovery: Dict
    response_contract: Dict


def create_agent_app(
    agent_id: str,
    description: str,
    discovery_backend: DiscoveryBackend,
    agent_logic_func: callable,
    max_depth: int = 2
) -> tuple:
    """
    Factory function to create a FastAPI app for an agent service
    
    Args:
        agent_id: Unique agent identifier
        description: Agent description for discovery
        discovery_backend: Discovery backend instance
        agent_logic_func: Async function that implements agent logic
                         Signature: async def agent_logic(goal: str, parameters: Dict) -> Dict
        max_depth: Maximum call depth
    
    Returns:
        (app, agent) tuple
    """
    
    app = FastAPI(title=f"{agent_id} Service")
    agent = A2AAgent(
        agent_id=agent_id,
        description=description,
        discovery_backend=discovery_backend,
        max_depth=max_depth
    )
    
    @app.on_event("startup")
    async def startup():
        await agent.register()
        logger.info(f"{agent_id} started and registered")
    
    @app.post("/a2a")
    async def handle_a2a_call(request: A2ARequestBody) -> Dict:
        """Handle incoming A2A calls"""
        try:
            a2a_request = A2ARequest.from_dict(request.dict())
        except Exception as e:
            logger.error(f"Invalid A2A request: {e}")
            raise HTTPException(status_code=400, detail=f"Invalid request: {e}")
        
        goal = a2a_request.payload.goal
        parameters = a2a_request.payload.parameters
        next_depth = a2a_request.metadata.call_depth + 1
        max_depth = a2a_request.metadata.max_depth
        trace_id = a2a_request.metadata.trace_id
        
        logger.info(f"{agent_id} received call: {goal} (depth={next_depth})")
        
        try:
            # Execute agent logic
            result = await agent_logic_func(
                goal=goal,
                parameters=parameters,
                agent=agent,
                next_depth=next_depth,
                max_depth=max_depth,
                trace_id=trace_id
            )
            
            latency_ms = 50  # Simplified for this example
            
            response = A2AResponse(
                metadata={
                    "responder_id": agent_id,
                    "call_id": a2a_request.metadata.call_id,
                    "trace_id": trace_id,
                    "timestamp": datetime.utcnow().isoformat()
                },
                status=ResponseStatus.SUCCESS,
                result=result,
                latency_ms=latency_ms
            )
            
            return response.to_dict()
        
        except Exception as e:
            logger.error(f"Error executing agent logic: {e}")
            response = A2AResponse(
                metadata={
                    "responder_id": agent_id,
                    "trace_id": trace_id
                },
                status=ResponseStatus.ERROR,
                errors=[{
                    "code": "execution_error",
                    "message": str(e)
                }]
            )
            return response.to_dict()
    
    @app.get("/health")
    async def health():
        return {"status": "healthy", "agent_id": agent_id}
    
    @app.get("/stats")
    async def stats():
        return agent.get_call_stats()
    
    return app, agent


# ============================================================================
# PART 4: CONCRETE AGENT EXAMPLES - FUNDER INTELLIGENCE SYSTEM
# ============================================================================

"""
Scenario: International Development Organization using AI to optimize funding strategy

Three agents collaborate to provide comprehensive funder intelligence:
1. Fundraising Agent: Individual high-net-worth investors, their interests, commitment levels
2. Business Development Agent: RFP (Request for Proposal) data, competing bids, funding priorities
3. Field Operations Agent: Local market conditions, project performance, capacity assessments

Use cases demonstrate how agents discover each other and cascade calls to build
comprehensive funding strategy intelligence.
"""


# Example 1: Fundraising Intelligence Agent
async def fundraising_logic(
    goal: str,
    parameters: Dict,
    agent: A2AAgent,
    next_depth: int,
    max_depth: int,
    trace_id: str
) -> Dict:
    """
    Fundraising Intelligence Agent
    
    Provides data on individual high-net-worth funders including:
    - Investment portfolio and interests
    - Past funding decisions and patterns
    - Current capacity and commitment level
    - Geographic and thematic focus areas
    """
    
    investor_id = parameters.get("investor_id")
    country = parameters.get("country")
    
    if "investor_profile" in goal.lower() or "angel_investor" in goal.lower():
        # Mock data for angel investor
        return {
            "investor_id": investor_id,
            "name": f"High Net Worth Individual {investor_id}",
            "total_portfolio": 150000000,  # $150M
            "available_capital": 25000000,  # $25M available
            "geographic_focus": ["East Africa", "Southeast Asia", "South Asia"],
            "thematic_interests": ["climate_action", "education", "health", "livelihoods"],
            "average_commitment_size": 5000000,  # $5M per deal
            "investment_stage_preference": "scaling_phase",
            "past_investments": [
                {
                    "country": "Kenya",
                    "amount": 3000000,
                    "year": 2023,
                    "sector": "climate_action"
                },
                {
                    "country": "Vietnam",
                    "amount": 2000000,
                    "year": 2023,
                    "sector": "education"
                }
            ],
            "risk_profile": "moderate",
            "decision_timeline_days": 90,
            "source": "fundraising-agent"
        }
    
    elif "investor_capacity" in goal.lower():
        # Return availability and capacity for specific country
        country = parameters.get("country")
        return {
            "investor_id": investor_id,
            "country": country,
            "available_capital": 5000000,
            "active_in_country": country in ["Kenya", "Vietnam", "Uganda"],
            "has_local_contacts": True,
            "local_experience_years": 8,
            "commitment_status": "actively_seeking",
            "source": "fundraising-agent"
        }
    
    elif "investor_interests" in goal.lower():
        # Match investor interests against specific criteria
        sector = parameters.get("sector")
        country = parameters.get("country")
        
        sector_match = sector in ["climate_action", "education", "health", "livelihoods"]
        country_match = country in ["East Africa", "Southeast Asia", "South Asia"]
        
        return {
            "investor_id": investor_id,
            "sector": sector,
            "country": country,
            "sector_interest_match": 0.9 if sector_match else 0.3,
            "geography_interest_match": 0.85 if country_match else 0.2,
            "overall_fit_score": 0.875 if (sector_match and country_match) else 0.25,
            "recommendation": "strong_candidate" if (sector_match and country_match) else "low_priority",
            "source": "fundraising-agent"
        }
    
    else:
        return {"error": f"Unknown goal: {goal}"}


# Example 2: Business Development Intelligence Agent
async def business_development_logic(
    goal: str,
    parameters: Dict,
    agent: A2AAgent,
    next_depth: int,
    max_depth: int,
    trace_id: str
) -> Dict:
    """
    Business Development Intelligence Agent
    
    Tracks RFP (Request for Proposal) bids, competitive landscape:
    - Active RFP opportunities
    - Competing organizations and their bids
    - Funder priorities and evaluation criteria
    - Win rates and decision timelines
    - Budget availability by sector/region
    """
    
    if "rfp_opportunities" in goal.lower():
        country = parameters.get("country")
        sector = parameters.get("sector")
        
        return {
            "country": country,
            "sector": sector,
            "active_rfps": [
                {
                    "rfp_id": "RFP-2025-KE-001",
                    "funder": "Global Climate Fund",
                    "budget": 50000000,
                    "deadline": "2025-03-15",
                    "sector": "climate_action",
                    "country": "Kenya",
                    "num_competitors": 12,
                    "our_estimated_win_probability": 0.35
                },
                {
                    "rfp_id": "RFP-2025-VN-002",
                    "funder": "World Education Initiative",
                    "budget": 20000000,
                    "deadline": "2025-02-28",
                    "sector": "education",
                    "country": "Vietnam",
                    "num_competitors": 8,
                    "our_estimated_win_probability": 0.55
                }
            ],
            "source": "business-development-agent"
        }
    
    elif "competitive_landscape" in goal.lower():
        rfp_id = parameters.get("rfp_id")
        
        return {
            "rfp_id": rfp_id,
            "total_competitors": 12,
            "major_competitors": [
                {
                    "organization": "International Dev Corp",
                    "estimated_bid": 48000000,
                    "past_win_rate": 0.42,
                    "strength_areas": ["implementation", "local_presence"]
                },
                {
                    "organization": "Global Solutions Fund",
                    "estimated_bid": 45000000,
                    "past_win_rate": 0.38,
                    "strength_areas": ["innovation", "technology"]
                }
            ],
            "competitive_intensity": "high",
            "differentiation_factors": ["local_partnerships", "cost_efficiency", "impact_metrics"],
            "source": "business-development-agent"
        }
    
    elif "funder_priorities" in goal.lower():
        funder_name = parameters.get("funder_name")
        
        return {
            "funder_name": funder_name,
            "top_priorities": ["climate_resilience", "local_capacity_building", "measurable_impact"],
            "evaluation_criteria": {
                "technical_approach": 0.30,
                "cost_efficiency": 0.20,
                "implementation_timeline": 0.15,
                "local_partnerships": 0.15,
                "sustainability": 0.20
            },
            "average_decision_timeline_days": 120,
            "funding_available": 50000000,
            "typical_grant_size": 3000000,
            "source": "business-development-agent"
        }
    
    else:
        return {"error": f"Unknown goal: {goal}"}


# Example 3: Field Operations Intelligence Agent
async def field_operations_logic(
    goal: str,
    parameters: Dict,
    agent: A2AAgent,
    next_depth: int,
    max_depth: int,
    trace_id: str
) -> Dict:
    """
    Field Operations Intelligence Agent
    
    Provides local market intelligence:
    - Current project portfolio and performance
    - Local capacity and team capabilities
    - Market demand and gaps
    - Stakeholder relationships
    - Government/regulatory environment
    - Partner ecosystem
    """
    
    country = parameters.get("country")
    
    if "country_capacity" in goal.lower():
        return {
            "country": country,
            "office_established": True,
            "years_operating": 8,
            "team_size": 45,
            "local_partnerships": 12,
            "government_relationships_strength": "strong",
            "budget_execution_rate": 0.92,
            "staff_retention_rate": 0.88,
            "technical_expertise_areas": ["climate_adaptation", "smallholder_agriculture", "community_engagement"],
            "capacity_constraints": ["specialized_staff", "local_financing_tools"],
            "source": "field-operations-agent"
        }
    
    elif "project_performance" in goal.lower():
        return {
            "country": country,
            "active_projects": 8,
            "total_beneficiaries": 125000,
            "projects": [
                {
                    "name": "Climate Smart Agriculture Initiative",
                    "budget": 5000000,
                    "progress_percent": 65,
                    "beneficiaries": 35000,
                    "status": "on_track",
                    "sector": "climate_action"
                },
                {
                    "name": "Women's Vocational Training",
                    "budget": 2500000,
                    "progress_percent": 80,
                    "beneficiaries": 5000,
                    "status": "on_track",
                    "sector": "livelihoods"
                },
                {
                    "name": "Digital Learning Centers",
                    "budget": 3000000,
                    "progress_percent": 45,
                    "beneficiaries": 12000,
                    "status": "delayed_regulatory",
                    "sector": "education"
                }
            ],
            "overall_performance_rating": 0.82,
            "source": "field-operations-agent"
        }
    
    elif "local_demand" in goal.lower():
        sector = parameters.get("sector")
        
        return {
            "country": country,
            "sector": sector,
            "identified_gaps": [
                "Limited climate adaptation financing for smallholders",
                "Insufficient vocational training capacity",
                "Weak digital infrastructure in rural areas"
            ],
            "estimated_unmet_need": 85000000,
            "stakeholder_priorities": [
                "Food security and livelihoods",
                "Youth employment",
                "Climate resilience"
            ],
            "partner_landscape": {
                "ngo_partners": 28,
                "government_agencies": 5,
                "private_sector": 12,
                "research_institutions": 3
            },
            "source": "field-operations-agent"
        }
    
    elif "market_assessment" in goal.lower():
        # This would normally call competitive funders to understand market
        sector = parameters.get("sector")
        
        # Could call competitive funders agent for RFP data
        competitive_response = await agent.call_agent(
            goal="get_competitive_landscape",
            parameters={"country": country, "sector": sector},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        competitive_data = competitive_response.result if competitive_response.status == ResponseStatus.SUCCESS else {}
        
        return {
            "country": country,
            "sector": sector,
            "local_demand": 85000000,
            "competitive_funding_available": competitive_data.get("total_budget", 0),
            "market_opportunity": "high",
            "recommended_positioning": "climate_adaptation_innovation",
            "source": "field-operations-agent",
            "includes_competitive_intelligence": competitive_response.status == ResponseStatus.SUCCESS
        }
    
    else:
        return {"error": f"Unknown goal: {goal}"}


# ============================================================================
# Example Workflow: Funding Strategy Agent (demonstrates cascade calling)
# ============================================================================

async def funding_strategy_logic(
    goal: str,
    parameters: Dict,
    agent: A2AAgent,
    next_depth: int,
    max_depth: int,
    trace_id: str
) -> Dict:
    """
    Funding Strategy Agent - Example of how to use all three agents together
    
    This would be a separate agent that orchestrates calls to the three intelligence agents
    to build a comprehensive funding strategy.
    
    Note: This demonstrates the cascade calling pattern and depth limiting.
    """
    
    if "evaluate_funding_opportunity" in goal.lower():
        country = parameters.get("country")
        sector = parameters.get("sector")
        investor_id = parameters.get("investor_id")
        rfp_id = parameters.get("rfp_id")
        
        # Call 1: Get country capacity (depth 1)
        country_response = await agent.call_agent(
            goal="assess country capacity and local demand",
            parameters={"country": country, "sector": sector},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        country_data = country_response.result if country_response.status == ResponseStatus.SUCCESS else {}
        
        # Call 2: Get angel investor profile (depth 1)
        investor_response = await agent.call_agent(
            goal="get angel investor profile and capacity",
            parameters={"investor_id": investor_id, "country": country},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        investor_data = investor_response.result if investor_response.status == ResponseStatus.SUCCESS else {}
        
        # Call 3: Get competitive landscape (depth 1)
        # NOTE: If max_depth=2, we CAN call all three at depth 1
        # But if field_operations wants to call business_development, THAT would be depth 2
        competitive_response = await agent.call_agent(
            goal="get competitive landscape and RFP opportunities",
            parameters={"country": country, "sector": sector, "rfp_id": rfp_id},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        competitive_data = competitive_response.result if competitive_response.status == ResponseStatus.SUCCESS else {}
        
        # Evaluate the opportunity
        local_capacity = country_data.get("office_established", False)
        investor_interest_match = investor_data.get("overall_fit_score", 0)
        market_fit = 0.8 if competitive_data else 0.5
        
        overall_score = (local_capacity * 0.4 + investor_interest_match * 0.3 + market_fit * 0.3)
        
        return {
            "country": country,
            "sector": sector,
            "investor_id": investor_id,
            "rfp_id": rfp_id,
            "overall_opportunity_score": round(overall_score, 2),
            "recommendation": "pursue_aggressively" if overall_score > 0.75 else "evaluate_further" if overall_score > 0.5 else "deprioritize",
            "local_capacity_strength": "strong" if local_capacity else "developing",
            "investor_fit": investor_interest_match,
            "competitive_landscape": competitive_data.get("total_competitors", "unknown"),
            "depth_limit_impact": "none",  # All calls succeeded
            "source": "funding-strategy-agent"
        }
    
    elif "identify_funding_gaps" in goal.lower():
        country = parameters.get("country")
        
        # Get country intelligence
        country_response = await agent.call_agent(
            goal="identify local demand and capacity gaps",
            parameters={"country": country},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        country_data = country_response.result if country_response.status == ResponseStatus.SUCCESS else {}
        
        # Try to get competitive funding (would fail at depth 2 if max_depth=2 and we called country first)
        competitive_response = await agent.call_agent(
            goal="identify available competitive funding",
            parameters={"country": country},
            current_depth=next_depth,
            trace_id=trace_id
        )
        
        if competitive_response.status == ResponseStatus.DEPTH_LIMIT_EXCEEDED:
            logger.warning(f"Could not get competitive data due to depth limit - returning partial analysis")
            gaps = country_data.get("identified_gaps", [])
            unmet_need = country_data.get("estimated_unmet_need", 0)
            competitive_available = "unknown"
        else:
            gaps = country_data.get("identified_gaps", [])
            unmet_need = country_data.get("estimated_unmet_need", 0)
            competitive_available = competitive_response.result.get("funding_available", 0)
        
        return {
            "country": country,
            "identified_gaps": gaps,
            "estimated_unmet_need": unmet_need,
            "competitive_funding_available": competitive_available,
            "funding_gap": unmet_need - (competitive_available if isinstance(competitive_available, int) else 0),
            "opportunity_for_new_funders": "high" if isinstance(competitive_available, int) and (unmet_need > competitive_available) else "moderate",
            "depth_limit_encountered": competitive_response.status == ResponseStatus.DEPTH_LIMIT_EXCEEDED,
            "source": "funding-strategy-agent"
        }
    
    else:
        return {"error": f"Unknown goal: {goal}"}


# ============================================================================
# PART 5: MAIN - RUNNING THE SYSTEM
# ============================================================================

async def main():
    """
    Example of running the A2A system with funder intelligence agents
    """
    
    # Set up shared discovery backend
    discovery = MockDiscoveryBackend()
    
    # Create apps for each agent
    angel_app, angel_agent = create_agent_app(
        agent_id="fundraising-agent",
        description="Intelligence on individual high-net-worth investors: portfolio, interests, capacity, commitment levels",
        discovery_backend=discovery,
        agent_logic_func=fundraising_logic,
        max_depth=2
    )
    
    competitive_app, competitive_agent = create_agent_app(
        agent_id="business-development-agent",
        description="Intelligence on competitive funding landscape: RFP opportunities, competing bids, funder priorities, budget availability",
        discovery_backend=discovery,
        agent_logic_func=business_development_logic,
        max_depth=2
    )
    
    country_app, country_agent = create_agent_app(
        agent_id="field-operations-agent",
        description="Local market intelligence: project performance, team capacity, local demand, stakeholder relationships, partner ecosystem",
        discovery_backend=discovery,
        agent_logic_func=field_operations_logic,
        max_depth=2
    )
    
    print("""
    ============================================================================
    A2A Protocol System - Funder Intelligence Discovery Phase
    ============================================================================
    
    International Development Organization - Funding Strategy System
    
    Three agents collaborate to provide comprehensive funder intelligence:
    
    1. fundraising-agent:      http://localhost:8001
       - Investor profiles and interests
       - Portfolio and capacity data
       - Geographic and thematic focus areas
    
    2. business-development-agent:  http://localhost:8002
       - RFP opportunities and funding available
       - Competitive landscape analysis
       - Funder priorities and evaluation criteria
    
    3. field-operations-agent:       http://localhost:8003
       - Project performance metrics
       - Local team capacity and expertise
       - Market demand and gaps
       - Partner ecosystem
    
    ============================================================================
    To run this in practice:
    ============================================================================
    
    # Terminal 1: Fundraising Agent
    uvicorn a2a_protocol_implementation:angel_app --port 8001
    
    # Terminal 2: Business Development Agent
    uvicorn a2a_protocol_implementation:competitive_app --port 8002
    
    # Terminal 3: Field Operations Agent
    uvicorn a2a_protocol_implementation:country_app --port 8003
    
    ============================================================================
    Example API Calls:
    ============================================================================
    
    # Evaluate funding opportunity (country calls angel investors and competitive funders)
    curl -X POST http://localhost:8003/a2a \\
      -H "Content-Type: application/json" \\
      -d '{
        "version": "a2a-v1",
        "message_type": "agent_call",
        "metadata": {
          "caller_id": "funding-strategy-agent",
          "call_depth": 0,
          "max_depth": 2,
          "trace_id": "eval-001"
        },
        "payload": {
          "goal": "evaluate funding opportunity for climate project",
          "parameters": {
            "country": "Kenya",
            "sector": "climate_action",
            "investor_id": "INV-001",
            "rfp_id": "RFP-2025-KE-001"
          }
        },
        "discovery": {"method": "semantic_search"},
        "response_contract": {}
      }'
    
    # Get angel investor profile
    curl -X POST http://localhost:8001/a2a \\
      -H "Content-Type: application/json" \\
      -d '{
        "version": "a2a-v1",
        "message_type": "agent_call",
        "metadata": {
          "caller_id": "funding-strategy-agent",
          "call_depth": 0,
          "max_depth": 2,
          "trace_id": "inv-001"
        },
        "payload": {
          "goal": "get angel investor profile",
          "parameters": {"investor_id": "INV-001"}
        },
        "discovery": {"method": "registry_lookup"},
        "response_contract": {}
      }'
    
    # Get competitive RFP opportunities
    curl -X POST http://localhost:8002/a2a \\
      -H "Content-Type: application/json" \\
      -d '{
        "version": "a2a-v1",
        "message_type": "agent_call",
        "metadata": {
          "caller_id": "field-operations-agent",
          "call_depth": 0,
          "max_depth": 2,
          "trace_id": "rfp-001"
        },
        "payload": {
          "goal": "identify RFP opportunities",
          "parameters": {
            "country": "Kenya",
            "sector": "climate_action"
          }
        },
        "discovery": {"method": "semantic_search"},
        "response_contract": {}
      }'
    
    # Check agent health
    curl http://localhost:8001/health
    
    # Get call statistics
    curl http://localhost:8001/stats
    
    ============================================================================
    Discovery Phase Insights (3 months):
    ============================================================================
    
    This system will log all agent-to-agent calls, enabling analysis of:
    
    1. Which agents teams actually want to call together
       - Funding strategy workflows
       - Investment decision patterns
       - Market opportunity identification
    
    2. Where cascade calling hits depth limits
       - When country office wants to call both angel investors AND competitive funders
       - Attempts at 3+ level deep reasoning chains
    
    3. Cross-organization calling patterns
       - Which country offices call which types of investors
       - Regional vs. global funding strategies
       - Sector-specific intelligence flows
    
    4. Latency and performance characteristics
       - Average response times by agent
       - Bottlenecks in funding intelligence gathering
       - Discovery effectiveness
    
    After 3 months, this data informs orchestrator design for production:
    - Should country offices have direct access to all investors?
    - What caching strategy improves competitive intelligence?
    - How should real-time RFP updates flow through the system?
    
    ============================================================================
    """)
    
    # For demonstration, show example call log entries
    print("\nExample successful call (Angel Investor lookup):")
    success_log = {
        "timestamp": datetime.utcnow().isoformat(),
        "caller": "funding-strategy-agent",
        "goal": "get angel investor profile",
        "target": "fundraising-agent",
        "depth": 0,
        "status": "success",
        "latency_ms": 42.5,
        "error": None
    }
    print(json.dumps(success_log, indent=2))


In [ ]:
print("\nExample depth-limited call (Country wants to call both investors AND competitors):")
    depth_limited_log = {
        "timestamp": datetime.utcnow().isoformat(),
        "caller": "field-operations-agent",
        "goal": "identify matching investors for climate opportunities",
        "target": None,
        "depth": 2,
        "status": "depth_limit_exceeded",
        "latency_ms": None,
        "error": {
            "code": "depth_limit_exceeded",
            "message": "Cannot call fundraising-agent - depth limit 2 reached",
            "current_depth": 2,
            "max_depth": 2,
            "workflow_impact": "unable_to_cross_match_investor_interests_with_local_opportunities"
        }
    }
    print(json.dumps(depth_limited_log, indent=2))


In [ ]:
if __name__ == "__main__":
    asyncio.run(main())
